# Lecture 9: Reinforcement Learning


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

np.random.seed(42)
print("Libraries loaded.")


## 1. Markov Decision Process (Ch. 19.1)

MDP components:
- **S**: State space
- **A**: Action space
- **P(s'|s,a)**: Transition probabilities
- **R(s,a)**: Reward function
- **γ**: Discount factor (0<γ<1)

**Goal:** Find a policy π that maximizes the cumulative discounted reward.


In [ ]:
# Grid World environment: 4x4 grid, start (0,0) → goal (3,3)
class GridWorld:
    def __init__(self, size=4):
        self.size = size
        self.start = (0, 0)
        self.goal  = (size-1, size-1)
        self.obstacles = {(1, 1), (2, 1), (1, 2)}
        self.reset()
    
    def reset(self):
        self.state = self.start
        return self.state
    
    def step(self, action):
        # 0=up 1=down 2=left 3=right
        moves = {0: (-1,0), 1: (1,0), 2: (0,-1), 3: (0,1)}
        dr, dc = moves[action]
        r, c = self.state
        nr, nc = max(0,min(self.size-1, r+dr)), max(0,min(self.size-1, c+dc))
        if (nr, nc) in self.obstacles:
            nr, nc = r, c  # hit obstacle
        self.state = (nr, nc)
        if self.state == self.goal:
            return self.state, 10.0, True  # goal
        return self.state, -0.1, False  # normal step
    
    def render(self, Q=None):
        grid = np.zeros((self.size, self.size))
        for obs in self.obstacles: grid[obs] = -1
        grid[self.goal] = 2
        grid[self.start] = 1
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        ax = axes[0]
        cmap = plt.cm.RdYlGn
        ax.imshow(grid, cmap=cmap, vmin=-1, vmax=2)
        
        for i in range(self.size):
            for j in range(self.size):
                if (i,j) in self.obstacles:
                    ax.text(j, i, 'X', ha='center', va='center', fontsize=16, color='black', fontweight='bold')
                elif (i,j) == self.goal:
                    ax.text(j, i, 'G', ha='center', va='center', fontsize=16, color='white', fontweight='bold')
                elif (i,j) == self.start:
                    ax.text(j, i, 'S', ha='center', va='center', fontsize=14, color='white')
        
        ax.set_title("Grid World Environment")
        ax.set_xticks(range(self.size)); ax.set_yticks(range(self.size))
        
        if Q is not None:
            ax2 = axes[1]
            action_symbols = ['^', 'v', '<', '>']
            v_vals = np.zeros((self.size, self.size))
            for i in range(self.size):
                for j in range(self.size):
                    if (i,j) not in self.obstacles:
                        best_a = np.argmax(Q[(i,j)])
                        v_vals[i,j] = Q[(i,j)][best_a]
                        ax2.text(j, i, action_symbols[best_a], ha='center', va='center',
                                fontsize=18, color='white', fontweight='bold')
            
            ax2.imshow(v_vals, cmap='Blues')
            ax2.set_title("Learned Policy and V Values")
            ax2.set_xticks(range(self.size)); ax2.set_yticks(range(self.size))
        else:
            axes[1].axis('off')
        
        plt.tight_layout()
        plt.savefig("fig_nb09_gridworld.png", dpi=100, bbox_inches='tight')
        plt.show()

env = GridWorld(size=4)
env.render()
print("S=Start, G=Goal, X=Obstacle")


## 2. Q-Learning: Bellman Equation (Ch. 19.3)

**Bellman update rule:**
$$Q(s, a) \\leftarrow Q(s, a) + \\alpha \\left[r + \\gamma \\max_{a'} Q(s', a') - Q(s, a)\\right]$$

- `r + γ max Q(s',a')`: TD target
- `r + γ max Q - Q(s,a)`: TD error


In [ ]:
def q_learning(env, n_episodes=2000, alpha=0.3, gamma=0.95, epsilon=0.3, eps_decay=0.999):
    Q = defaultdict(lambda: np.zeros(4))
    rewards_log = []
    steps_log   = []
    
    for ep in range(n_episodes):
        state = env.reset()
        total_r = 0
        steps = 0
        eps = max(0.05, epsilon * (eps_decay ** ep))
        
        for _ in range(100):
            if np.random.rand() < eps:
                action = np.random.randint(4)
            else:
                action = np.argmax(Q[state])
            
            next_state, reward, done = env.step(action)
            
            # Bellman update
            td_target = reward + gamma * np.max(Q[next_state]) * (not done)
            Q[state][action] += alpha * (td_target - Q[state][action])
            
            state = next_state
            total_r += reward
            steps += 1
            if done: break
        
        rewards_log.append(total_r)
        steps_log.append(steps)
    
    return Q, rewards_log, steps_log

env = GridWorld(size=4)
Q_learned, rewards, steps = q_learning(env, n_episodes=3000)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
window = 100
smoothed_r = np.convolve(rewards, np.ones(window)/window, mode='valid')
smoothed_s = np.convolve(steps,   np.ones(window)/window, mode='valid')

axes[0].plot(smoothed_r, 'steelblue', lw=2)
axes[0].set_title(f"Total Reward per Episode (rolling average={window})")
axes[0].set_xlabel("Episode"); axes[0].set_ylabel("Total Reward"); axes[0].grid(alpha=0.3)

axes[1].plot(smoothed_s, 'crimson', lw=2)
axes[1].set_title("Steps to Reach Goal")
axes[1].set_xlabel("Episode"); axes[1].set_ylabel("Step count"); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("fig_nb09_qlearning.png", dpi=100, bbox_inches='tight')
plt.show()

env.render(Q_learned)


## 3. Epsilon-Greedy Exploration-Exploitation Tradeoff

- **ε=1**: Fully random (exploration)
- **ε=0**: Fully greedy (exploitation)
- **ε decay**: Transition from exploration to exploitation during training


In [ ]:
epsilons = np.array([1.0, 0.8, 0.5, 0.2, 0.05])
n_actions = 4
q_values = np.array([0.2, 0.8, 0.3, 0.5])  # Action 1 is best

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for eps in epsilons:
    n_sim = 5000
    choices = []
    for _ in range(n_sim):
        if np.random.rand() < eps:
            choices.append(np.random.randint(n_actions))
        else:
            choices.append(np.argmax(q_values))
    counts = np.bincount(choices, minlength=n_actions) / n_sim
    axes[0].bar(np.arange(n_actions) + (list(epsilons).index(eps))*0.15 - 0.3,
                counts, width=0.15, label=f"eps={eps}")

axes[0].set_title("Epsilon-Greedy: Action Selection Distribution")
axes[0].set_xlabel("Action (1 = optimal)"); axes[0].set_ylabel("Selection probability")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3, axis='y')
axes[0].set_xticks(range(n_actions))

ep_range = np.arange(1000)
decays = {0.999: "Slow decay", 0.995: "Medium decay", 0.99: "Fast decay"}
for decay, label in decays.items():
    eps_curve = np.maximum(0.05, 1.0 * decay**ep_range)
    axes[1].plot(ep_range, eps_curve, lw=2, label=label)

axes[1].set_title("Epsilon Decay Strategies")
axes[1].set_xlabel("Episode"); axes[1].set_ylabel("Epsilon")
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].axhline(0.05, color='gray', linestyle='--', lw=1, label="Min epsilon")

plt.tight_layout()
plt.savefig("fig_nb09_epsilon.png", dpi=100, bbox_inches='tight')
plt.show()


## 4. Policy Gradient — REINFORCE (Ch. 19.5)

Instead of value-based methods, the **policy** is learned directly:

$$\\nabla_\\theta J(\\theta) = \\mathbb{E}_\\pi \\left[G_t \\nabla_\\theta \\log \\pi_\\theta(a_t|s_t)\\right]$$

`G_t = R_t + γR_{t+1} + γ²R_{t+2} + ...` = episodic return


In [ ]:
def softmax_policy(state_feat, theta):
    logits = state_feat @ theta
    logits -= logits.max()
    probs = np.exp(logits)
    probs /= probs.sum()
    return probs

class SimpleEnv1D:
    def __init__(self):
        self.pos = 0.0
        self.target = 1.0
        self.steps = 0
    
    def reset(self):
        self.pos = np.random.uniform(-1, 0)
        self.steps = 0
        return np.array([self.pos, self.target - self.pos])
    
    def step(self, action):
        move = 0.1 if action == 1 else -0.1
        self.pos += move + np.random.randn()*0.02
        self.steps += 1
        state = np.array([self.pos, self.target - self.pos])
        dist = abs(self.pos - self.target)
        reward = -dist
        done = dist < 0.1 or self.steps >= 50
        if dist < 0.1: reward = 5.0
        return state, reward, done

def reinforce(n_episodes=1000, gamma=0.95, lr=0.05):
    env_r = SimpleEnv1D()
    n_features = 2; n_actions = 2
    theta = np.random.randn(n_features, n_actions) * 0.1
    all_rewards = []
    
    for ep in range(n_episodes):
        states, actions, rewards = [], [], []
        state = env_r.reset()
        
        for _ in range(50):
            probs = softmax_policy(state, theta)
            action = np.random.choice(n_actions, p=probs)
            next_state, reward, done = env_r.step(action)
            states.append(state); actions.append(action); rewards.append(reward)
            state = next_state
            if done: break
        
        # Returns
        G = 0
        returns = []
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = np.array(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        
        # Gradient update
        for i, (s, a, g) in enumerate(zip(states, actions, returns)):
            probs = softmax_policy(s, theta)
            grad_log = np.zeros((len(s), n_actions))
            grad_log[:, a] = s
            for k in range(n_actions):
                grad_log[:, k] -= s * probs[k]
            theta += lr * g * grad_log
        
        all_rewards.append(sum(rewards))
    
    return theta, all_rewards

theta_pg, pg_rewards = reinforce(n_episodes=1500)

window = 50
smoothed_pg = np.convolve(pg_rewards, np.ones(window)/window, mode='valid')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(pg_rewards, color='gray', lw=1, alpha=0.4)
ax.plot(smoothed_pg, color='crimson', lw=2, label=f"Rolling average ({window})")
ax.set_title("REINFORCE (Policy Gradient) Training")
ax.set_xlabel("Episode"); ax.set_ylabel("Total Reward")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("fig_nb09_reinforce.png", dpi=100, bbox_inches='tight')
plt.show()
print(f"Last 100 episodes mean reward: {np.mean(pg_rewards[-100:]):.2f}")


## 5. Deep RL Algorithm Summary

| Algorithm | Method | Network | Usage |
|---|---|---|---|
| **Q-Learning** | Value-based | Q-network | Tabular / small space |
| **DQN** | Value-based + DNN | CNN + FC | Atari games |
| **REINFORCE** | Policy gradient | Policy network | Continuous action |
| **Actor-Critic** | Hybrid | Actor + Critic | Robotics, control |
| **PPO** | Policy gradient + clip | Actor + Critic | Modern RL default |

## 6. Summary

| Concept | Description |
|---|---|
| **MDP** | State-action-reward-policy loop |
| **Q(s,a)** | State-action value function |
| **Bellman** | Recursive Q update rule |
| **Epsilon-greedy** | Exploration-exploitation balance strategy |
| **Policy gradient** | Directly optimize policy parameters |
| **Return G_t** | Discounted cumulative reward |
